In [0]:
%pip install beautifulsoup4

In [0]:
from bs4 import BeautifulSoup

try:
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# 1. Define URL
base_page_url = "https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open"

# -------------------------------------------------------------------------
# 2. Azure Storage Paths (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "" # <-- PASTE Name here (e.g., "adlsdubaisakib")
storage_account_key = "" # <-- PASTE Key here (e.g., "b8+mS9...")

In [0]:
# Connect to ADLS
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
# ✨ AUTO-CREATE FIX: This creates the 'bronze' container if it's missing
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# Define Folders
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"

In [0]:
# 3. Dynamic Link Finder
def get_dynamic_download_url():
    print(f"🔍 Scraping {base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
 
    response = requests.get(base_page_url, headers=headers)
    response.raise_for_status()
 
    soup = BeautifulSoup(response.content, 'html.parser')
 
    # Logic: Find link containing 'download' and 'transactions.csv'
    for link in soup.find_all('a', href=True):
        href = link['href']
        if "download" in href and "transactions.csv" in href:
            final_url = f"https://www.dubaipulse.gov.ae{href}" if href.startswith("/") else href
            print(f"✅ Found dynamic link: {final_url}")
            return final_url
    raise Exception("❌ Could not find the download link on the page.")
    
    print("Configuration Loaded & Auto-Create Enabled.")


In [0]:
def cleanup_old_files(days_to_keep=14):
    print("🧹 Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
 
    try:
        files = dbutils.fs.ls(archive_zone)
        for f in files:
            # Expected format: YYYY-MM-DD.csv
            filename = f.name.replace(".csv", "")
            try:
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date < cutoff_date:
                    print(f" 🗑️ Deleting: {f.name}")
                    dbutils.fs.rm(f.path)
            except ValueError:
                continue
    except Exception:
        print(" ℹ️ Archive is empty or does not exist yet.")
    
cleanup_old_files()

In [0]:
# CELL 4: Archival Logic
# -------------------------------------------------------------------------
def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"
    
    try:
        # Check if file exists
        dbutils.fs.ls(current_file_path)
        
        # Rename to YYYY-MM-DD.csv
        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"
        
        print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
        dbutils.fs.mv(current_file_path, archive_path)
        
    except Exception:
        print("ℹ️ No 'current.csv' found in landing. First run?")

archive_current_file()

In [0]:
# CELL 5: Download Snapshot (Robust Version with Retries)
# -------------------------------------------------------------------------
import os
import time

def download_snapshot_with_retry(max_retries=3):
    fresh_url = get_dynamic_download_url()
    local_path = "/tmp/current.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading to driver node...")
            
            # 30-second timeout, Stream=True
            with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    # 10MB Chunks
                    for chunk in r.iter_content(chunk_size=10*1024*1024):
                        if chunk: 
                            f.write(chunk)
            
            print("   ✅ Download complete.")
            break # Exit loop on success
            
        except Exception as e:
            print(f"   ⚠️ Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                print("   ⏳ Waiting 5 seconds...")
                time.sleep(5)
            else:
                print("❌ All retries failed.")
                raise e

    # Upload to Data Lake
    if os.path.exists(local_path):
        print(f"⬆️ Uploading to Bronze Landing Zone...")
        dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
        os.remove(local_path)
        print("✅ Success! 'current.csv' is updated in the Data Lake.")

# Run it
download_snapshot_with_retry()